# RAG EWS — satu notebook, step by step

**Step 1 — Ingest:** fondasi valid (`count == 86`, query GCS balik `CHUNK0002`).
**Step 2 — RAG end-to-end:** top-3 → LLM kecil + sitasi, termasuk uji tolak pertanyaan luar konteks.
**Step 3 — Kalkulator EWS + routing:** `/check ...` (angka vital) → hitung skor rule-based; tanya konsep → RAG.
**Step 4 — Penutup:** checklist demo + cara simpan hasil.

Jalankan berurutan (Runtime > Run all).

In [ ]:
%pip install -q chromadb

In [ ]:
import json, urllib.request

URLS = [
    "https://raw.githubusercontent.com/iaanne/nlp-rag-kelompok6/ian/data/EWS_v5_USONLY_final_QA.jsonl",
    "https://huggingface.co/datasets/Paxrad/EWS_v5_USONLY_final/resolve/main/EWS_v5_USONLY_final_QA.jsonl",
]

for u in URLS:
    try:
        urllib.request.urlretrieve(u, "/content/EWS_v5_USONLY_final_QA.jsonl")
        print("downloaded from:", u)
        break
    except Exception as e:
        print("failed:", u, "->", e)

rows = [json.loads(l) for l in open("/content/EWS_v5_USONLY_final_QA.jsonl", encoding="utf-8")]
print("rows:", len(rows))
print(rows[0])

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="/content/chroma_db")
col = client.get_or_create_collection("ews_v5")

if col.count() == 0:
    col.add(
        ids=[r["id"] for r in rows],
        documents=[r["question"] + "\n" + r["answer"] for r in rows],
        metadatas=[{"chunk_id": r["chunk_id"], "question_type": r.get("question_type", "")} for r in rows],
    )

print("count:", col.count())
assert col.count() == 86, "expected 86 docs"

In [ ]:
res = col.query(query_texts=["Which chapter discusses the Glasgow Coma Scale?"], n_results=3)
for i, m, d in zip(res["ids"][0], res["metadatas"][0], res["distances"][0]):
    print(f"{i} | {m['chunk_id']} | dist={d:.4f}")

top_chunks = [m["chunk_id"] for m in res["metadatas"][0]]
assert "EWS.v5.CHUNK0002" in top_chunks, "retrieval check failed"
print("RETRIEVAL OK")

## Step 2 — Retriever + LLM (RAG end-to-end)

Pola RAG yang dipakai: pertanyaan → ambil top-3 dokumen → ditempel ke prompt → LLM kecil jawab **hanya** dari konteks + tulis sitasi `[chunk_id]`.

**Kriteria lolos:** jawaban 3 pertanyaan medis sesuai konteks + sitasinya benar, dan pertanyaan jebakan di luar konteks (ibukota Prancis) **ditolak**.

In [ ]:
%pip install -q transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # kecil, muat di Colab gratis
tok = AutoTokenizer.from_pretrained(MODEL)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="auto")
except TypeError:  # transformers lama pakai nama arg berbeda
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto")
print("model loaded on:", model.device)


def retrieve(q, k=3):
    r = col.query(query_texts=[q], n_results=k)
    return list(zip(r["documents"][0], r["metadatas"][0]))


def ask(q, k=3):
    docs = retrieve(q, k)
    ctx = "\n\n".join(f"[{m['chunk_id']}] {d}" for d, m in docs)
    messages = [
        {"role": "system", "content": (
            "You are a field triage assistant. Answer ONLY using the context below. "
            "Cite sources as [chunk_id]. If the answer is not in the context, say you do not know.")},
        {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {q}"},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("Q:", q, "\n\nA:", gen.strip())
    print("\nSources:", ", ".join(m["chunk_id"] for _, m in docs))
    print("\n" + "=" * 60 + "\n")

In [ ]:
ask("Which chapter discusses the Glasgow Coma Scale?")
ask("What is the initial hourly crystalloid rate for a 40% burn?")
ask("What is the role of the Chief Surgical Triage Officer?")
ask("What is the capital of France?")  # jebakan: harus menolak

## Step 3 — Kalkulator EWS rule-based + routing

Kenapa terpisah dari RAG: dataset berisi QA naratif, **bukan** tabel skor — jadi angka EWS dihitung manual dari tabel NEWS2 yang diprogram di bawah, bukan di-retrieve.

> ⚠️ Demo edukasi, bukan alat medis klinis.

Aturan routing: input diawali `/check` (angka vital) → kalkulator; selain itu → RAG (`ask`).

In [ ]:
def s_rr(x):
    if x <= 8: return 3
    if x <= 11: return 1
    if x <= 20: return 0
    if x <= 24: return 2
    return 3

def s_spo2(x):
    if x <= 91: return 3
    if x <= 93: return 2
    if x <= 95: return 1
    return 0

def s_temp(x):
    if x <= 35.0: return 3
    if x <= 36.0: return 1
    if x <= 38.0: return 0
    if x <= 39.0: return 1
    return 2

def s_sys(x):
    if x <= 90: return 3
    if x <= 100: return 2
    if x <= 110: return 1
    if x <= 219: return 0
    return 3

def s_hr(x):
    if x <= 40: return 3
    if x <= 50: return 1
    if x <= 90: return 0
    if x <= 110: return 1
    if x <= 130: return 2
    return 3

def s_conscious(avpu=None, gcs=None):
    if gcs is not None:
        return 0 if gcs == 15 else 3
    return 0 if (avpu or "A").upper() == "A" else 3

def ews(v):
    """v: dict HR, SYS, RR, SPO2, T, AVPU/GCS, O2(air/oxygen)."""
    parts = {
        "RR": s_rr(v["RR"]),
        "SpO2": s_spo2(v["SPO2"]),
        "O2": 0 if v.get("O2", "air").lower() == "air" else 2,
        "Temp": s_temp(v["T"]),
        "SYS": s_sys(v["SYS"]),
        "HR": s_hr(v["HR"]),
        "Conscious": s_conscious(v.get("AVPU"), v.get("GCS")),
    }
    total = sum(parts.values())
    single3 = any(s == 3 for s in parts.values())
    if total >= 7:
        band, advice = "HIGH", "Tim emergency segera (resus)."
    elif total >= 5:
        band, advice = "MEDIUM", "Review medis segera."
    elif total >= 3 or single3:
        band, advice = "LOW-MEDIUM", "Perawat senior menilai; pertimbangkan eskalasi."
    else:
        band, advice = "LOW", "Pemantauan rutin."
    return total, band, parts, advice


def show_ews(v):
    total, band, parts, advice = ews(v)
    print(f"EWS={total} [{band}]")
    for k, s in parts.items():
        print(f"  {k}: {s}")
    print("Advice:", advice)
    return parts


print("--- pasien stabil ---")
show_ews({"HR": 72, "SYS": 120, "RR": 16, "SPO2": 98, "T": 36.8, "AVPU": "A", "O2": "air"})
print("\n--- pasien medium ---")
show_ews({"HR": 95, "SYS": 105, "RR": 22, "SPO2": 95, "T": 37.5, "AVPU": "A", "O2": "air"})
print("\n--- pasien gawat ---")
p_high = show_ews({"HR": 125, "SYS": 88, "RR": 29, "SPO2": 90, "T": 39.5, "AVPU": "V", "O2": "oxygen"})

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.barh(list(p_high.keys()), list(p_high.values()))
plt.xlabel("Skor per parameter")
plt.title("Visualisasi risiko EWS — pasien gawat")
plt.tight_layout()
plt.show()

### Routing: quick command vs tanya konsep

Format perintah: `/check HR=.. SYS=.. RR=.. SPO2=.. T=.. AVPU=.. O2=..` (boleh juga `BP=85/60` untuk sistolik, atau `GCS=..` pengganti AVPU).

In [ ]:
import re

def parse_check(cmd):
    kv = {k.upper(): v for k, v in re.findall(r"(\w+)\s*=\s*([\w./]+)", cmd)}
    v = {"HR": 70, "SYS": 120, "RR": 16, "SPO2": 98, "T": 36.8, "AVPU": "A", "O2": "air"}
    if "HR" in kv: v["HR"] = float(kv["HR"])
    if "SYS" in kv: v["SYS"] = float(kv["SYS"])
    if "BP" in kv: v["SYS"] = float(kv["BP"].split("/")[0])
    if "RR" in kv: v["RR"] = float(kv["RR"])
    if "SPO2" in kv: v["SPO2"] = float(kv["SPO2"])
    if "T" in kv: v["T"] = float(kv["T"])
    if "TEMP" in kv: v["T"] = float(kv["TEMP"])
    if "AVPU" in kv: v["AVPU"] = kv["AVPU"]
    if "GCS" in kv: v["GCS"] = int(float(kv["GCS"]))
    if "O2" in kv: v["O2"] = kv["O2"]
    return v


def route(text):
    if text.strip().lower().startswith("/check"):
        show_ews(parse_check(text))
    else:
        ask(text)


route("/check HR=125 SYS=88 RR=29 SPO2=90 T=39.5 AVPU=V O2=oxygen")
route("What should be done with nonemergent patients in a mass casualty situation?")

## Step 4 — Penutup: checklist demo + simpan hasil

**Checklist sebelum bilang selesai:**
- [ ] Step 1 hijau: `count == 86`, GCS balik `CHUNK0002`
- [ ] Step 2 hijau: 3 jawaban medis benar + bersitasi, jebakan ditolak
- [ ] Step 3 hijau: pasien stabil LOW, medium MEDIUM, gawat HIGH + grafik muncul
- [ ] Routing: `/check ...` keluar skor, tanya konsep keluar jawaban RAG

**Simpan hasil:** di Colab File > Download `.ipynb` (atau biarkan di Drive), lalu di laptop: `git add notebooks/`, commit, push ke `ian`.

**Next (di luar notebook ini):** bungkus jadi web (Streamlit/FastAPI) dengan panel jawaban + panel rujukan chunks + panel skor EWS; tambah korpus K3 Indonesia sebagai collection kedua.